In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import os


In [9]:
datetime(2020,10,27)

datetime.datetime(2020, 10, 27, 0, 0)

In [29]:
#Initial code from ChatGPT



# -----------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------

# URL to SPC daily log (change date as needed, here fixed to your link)
URL = "https://www.spc.noaa.gov/exper/reports/data/rpts/201027.log"

# Which storm section you want (exactly match header)
SECTION_NAME = "FREEZING RAIN"

# -----------------------------------------------------------------
# Download text
# -----------------------------------------------------------------

print("Downloading SPC LSR log...")
resp = requests.get(URL)
resp.raise_for_status()
text = resp.text.splitlines()

# -----------------------------------------------------------------
# Parse by section
# -----------------------------------------------------------------

# Find the start of the section header
section_header = None
for idx, line in enumerate(text):
    # Header lines contain SECTION_NAME between asterisks
    #print(line)
    if f"***{SECTION_NAME}***" in line:
        section_header = idx
        break

if section_header is None:
    raise RuntimeError(f"Section {SECTION_NAME} not found in log")

# Next line after header contains the field names (after the "***...***" line)
header_line = text[section_header + 2]
# Clean header: remove leading/trailing asterisks and whitespace
fields_str = header_line.split("***")[2].strip() if "***" in header_line else header_line
# We expect something like: " #|TIME|TYPE|E/M/U|MAG|..."
fields = [f.strip() for f in fields_str.split("|")]

# Collect entries below this header until a blank or next section
rows = []
for line in text[section_header+4:]:
    if not line.strip():
        # Stop on blank line
        break
    # Lines that look like "001|..." are data
    parts = [p.strip() for p in line.split("|")]
    # Skip anything that doesn't match field count
    if len(parts) == len(fields):
        rows.append(parts)
    else:
        # Might be a wrap or non-data; skip or break
        continue

# -----------------------------------------------------------------
# Build DataFrame
# -----------------------------------------------------------------

df = pd.DataFrame(rows, columns=fields)

# -----------------------------------------------------------------
# Convert some columns to numeric if possible
# -----------------------------------------------------------------

# Example: magnitude, lat, lon
for col in ["MAG", "LAT", "LON"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# If you have an issuance timestamp field, convert if desired
if "ISSUANCE" in df.columns:
    df["ISSUANCE"] = pd.to_numeric(df["ISSUANCE"], errors="coerce")

print(f"Parsed {len(df)} rows of {SECTION_NAME} data")

# -----------------------------------------------------------------
# Show or save
# -----------------------------------------------------------------

print(df.head())

Parsed 49 rows of FREEZING RAIN data
     #  TIME TYPE E/M/U  MAG DISTANCE DIRECTION          CITY    COUNTY  ST  \
0  363  1215   FR     E   10        1       WNW  GARDEN PLAIN  SEDGWICK  KS   
1  364  1300   FR     E    5        0                EASTLAND  EASTLAND  TX   
2  365  1300   FR     E    5        1         E  BRECKENRIDGE  STEPHENS  TX   
3  366  1354   FR     M   10        7       SSW      WINFIELD    COWLEY  KS   
4  367  1420   FR     E   25        0                 MARAMEC    PAWNEE  OK   

    LAT   LON           SOURCE  \
0  3767  9769  BROADCAST MEDIA   
1  3240  9882  LAW ENFORCEMENT   
2  3276  9889  LAW ENFORCEMENT   
3  3718  9704             ASOS   
4  3624  9668  BROADCAST MEDIA   

                                              REMARK OFFICE    ISSUANCE  \
0  MIXTURE OF FREEZING RAIN AND SLEET... ICE ACCU...    ICT  1603802340   
1  LOCAL LAW ENFORCEMENT REPORTING A LIGHT GLAZE ...    FWD  1603816020   
2  LOCAL LAW ENFORCEMENT REPORTING A LIGHT GLAZE ...    FW

In [30]:
df

,#,TIME,TYPE,E/M/U,MAG,DISTANCE,DIRECTION,CITY,COUNTY,ST,LAT,LON,SOURCE,REMARK,OFFICE,ISSUANCE,FATALITIES,INJURIES
0,363,1215,FR,E,10,1,WNW,GARDEN PLAIN,SEDGWICK,KS,3767,9769,BROADCAST MEDIA,MIXTURE OF FREEZING RAIN AND SLEET... ICE ACCU...,ICT,1603802340,0,0
1,364,1300,FR,E,5,0,,EASTLAND,EASTLAND,TX,3240,9882,LAW ENFORCEMENT,LOCAL LAW ENFORCEMENT REPORTING A LIGHT GLAZE ...,FWD,1603816020,0,0
2,365,1300,FR,E,5,1,E,BRECKENRIDGE,STEPHENS,TX,3276,9889,LAW ENFORCEMENT,LOCAL LAW ENFORCEMENT REPORTING A LIGHT GLAZE ...,FWD,1603816200,0,0
3,366,1354,FR,M,10,7,SSW,WINFIELD,COWLEY,KS,3718,9704,ASOS,ICE ACCUMULATION MEASURED BY THE ASOS AT STROT...,ICT,1603807320,0,0
4,367,1420,FR,E,25,0,,MARAMEC,PAWNEE,OK,3624,9668,BROADCAST MEDIA,ESTIMATED ICE ACCUMULATION ON TREES BY STORM T...,TSA,1603809000,0,0
5,368,1420,FR,E,25,0,,FAIRFAX,OSAGE,OK,3657,9670,BROADCAST MEDIA,ESTIMATED ICE ACCUMULATION ON TREES BY STORM T...,TSA,1603809000,0,0
6,369,1455,FR,M,15,7,N,ARKANSAS CITY,COWLEY,KS,3717,9704,ASOS,STORM TOTAL ICE ACCUMULATION SINCE 7 AM MEASUR...,ICT,1603811220,0,0
7,370,1455,FR,M,9,6,WSW,DOWNTOWN WICHITA,SEDGWICK,KS,3765,9744,ASOS,STORM TOTAL ICE ACCUMULATION SINCE 7 AM MEASUR...,ICT,1603811520,0,0
8,371,1455,FR,E,7,3,ESE,BEL AIRE,SEDGWICK,KS,3775,9722,ASOS,STORM TOTAL ICE ACCUMULATION SINCE 7 AM MEASUR...,ICT,1603811700,0,0
9,372,1455,FR,M,9,3,E,HUTCHINSON,RENO,KS,3807,9785,ASOS,STORM TOTAL ICE ACCUMULATION SINCE 7 AM MEASUR...,ICT,1603812000,0,0
